# Notebook 15 — Reception-Threshold Sensitivity Analysis
## FYP: Adaptive Explainable Ensemble for Pre-Launch Steam Game Reception Prediction
### Heshan Ratnaweera | IIT Sri Lanka | W2082289 | 2026

**Purpose:** Test whether the system's performance and conclusions hold at reception-label
thresholds other than the 75% cutoff used throughout the project. This answers Reviewer 1's
comment: *"The 75% positive review threshold used to define successful reception is a design
choice. The authors should provide additional justification or sensitivity analysis using
alternative thresholds."*

**Important distinction from NB09/NB10:** those notebooks tested the minimum **review-count**
threshold (100 vs. 50 reviews required for inclusion). This notebook tests the **reception-label**
threshold (currently ≥75% positive = "Well Received") — a completely different design choice that
has never been tested at alternative values anywhere in NB01–NB14.

**This notebook DOES need the raw dataset.** `pct_pos_total` (the raw percentage used to build
the label) is marked `LABEL SOURCE` in NB01's column decision matrix and is not retained as a
feature in `games_features_t4.csv`. We reconstruct it from `data/raw/Dataset-2.csv` using the
exact same filtering logic as NB01, then verify it reproduces the existing 75%-threshold label
exactly before testing anything new — this is a critical safety check, explained in Section 3.

**This notebook is READ-ONLY with respect to existing files.** It loads the raw CSV and
`games_features_t4.csv`, but writes nothing back to either. New models trained here are only
kept in memory for evaluation, not saved over `model_d.pkl`.

**Inputs (read-only):**
- `data/raw/Dataset-2.csv`
- `data/processed/games_features_t4.csv`

**What stays fixed across all thresholds tested:** the CatBoost hyperparameters (depth=8,
learning_rate=0.05, iterations=1000, l2_leaf_reg=3, scale_pos_weight=0.5 — Model D's actual
config from NB04) and the T4 structured features themselves. Only the label changes. This
isolates the effect of the threshold decision from every other variable — hyperparameters are
deliberately NOT re-tuned per threshold, since re-tuning for each one would conflate "does the
threshold choice matter" with "did we search hard enough for each variant."

---
## Contents
1. Setup and Imports
2. Load Raw Dataset and Reconstruct the Filter
3. Verification — Does the Reconstructed 75% Label Match the Existing One?
4. Build Alternative-Threshold Labels
5. Retrain Model D at Each Threshold
6. Results Table and Class-Balance Comparison
7. Summary


## 1. Setup and Imports

In [1]:
import sys
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, roc_auc_score, accuracy_score

warnings.filterwarnings('ignore')

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.config import (
    TARGET_COL, RANDOM_STATE, FEATURES_T4_CSV, TIER_FEATURES,
    RAW_CSV, COL_LABEL, COL_FILTER, CATBOOST_PARAMS,
)

# Model D's exact config, from src.config — kept fixed across all thresholds tested.
CB_PARAMS = CATBOOST_PARAMS

print(f'Project root : {PROJECT_ROOT}')
print(f'Random state : {RANDOM_STATE}')


Project root : C:\Users\3214h\Documents\fyp-steam-reception
Random state : 42


## 2. Load Raw Dataset and Reconstruct the Filter

Reproduces NB01's exact filtering: remove sentinel `-1` rows in `pct_pos_total` and
`num_reviews_total`, then keep only games with `num_reviews_total >= 100`. This should reproduce
the 20,383-game working set reported throughout the project.

**Adjust the file path below if your raw CSV lives somewhere else or has a different filename.**


In [2]:
assert RAW_CSV.exists(), f'Not found: {RAW_CSV} — check the path matches your project layout.'

df_raw = pd.read_csv(RAW_CSV, low_memory=False)
print(f'Loaded raw: {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns')

assert COL_LABEL in df_raw.columns, f'{COL_LABEL} column not found — check raw CSV schema.'
assert COL_FILTER in df_raw.columns, f'{COL_FILTER} column not found — check raw CSV schema.'

# Remove sentinel -1 rows, then apply the >=100 review filter (matches NB01 exactly)
df_filtered = df_raw[
    (df_raw[COL_LABEL] != -1) &
    (df_raw[COL_FILTER] != -1) &
    (df_raw[COL_FILTER] >= 100)
].reset_index(drop=True)

print(f'After filtering: {df_filtered.shape[0]:,} rows')
print(f'Expected (per NB01): 20,383 rows')

if df_filtered.shape[0] != 20383:
    print()
    print('*** WARNING: row count does not match NB01. Check the filter logic above against')
    print('*** your actual NB01/NB02 preprocessing before trusting anything downstream.')
else:
    print('Row count matches — filter reconstruction looks correct.')


Loaded raw: 89,618 rows x 47 columns
After filtering: 20,383 rows
Expected (per NB01): 20,383 rows
Row count matches — filter reconstruction looks correct.


## 3. Verification — Does the Reconstructed 75% Label Match the Existing One?

**Do not skip this step.** Before testing any alternative threshold, we confirm that applying the
existing 75% rule to our reconstructed `pct_pos_total` produces the same class balance already
reported everywhere else in the project (71.8% Well Received / 28.2% Not Well Received). If this
does not match, the row order or filter between this notebook and NB01/NB02 has diverged, and
every result after this point would be silently wrong.


In [3]:
df_filtered['reception_label_75_reconstructed'] = (
    df_filtered[COL_LABEL] >= 75
).astype(int)

pct_well_received = df_filtered['reception_label_75_reconstructed'].mean() * 100
print(f'Reconstructed 75% threshold — class balance:')
print(f'  Well Received     : {pct_well_received:.1f}%')
print(f'  Not Well Received : {100 - pct_well_received:.1f}%')
print(f'  Expected (per NB01/paper): 71.8% / 28.2%')
print()

assert FEATURES_T4_CSV.exists(), f'Not found: {FEATURES_T4_CSV}'
df_t4 = pd.read_csv(FEATURES_T4_CSV)

if df_t4.shape[0] != df_filtered.shape[0]:
    print(f'*** WARNING: row count mismatch — games_features_t4.csv has {df_t4.shape[0]:,} rows, '
          f'reconstructed filter has {df_filtered.shape[0]:,} rows. DO NOT PROCEED until resolved.')
else:
    existing_pct = df_t4[TARGET_COL].mean() * 100
    print(f'games_features_t4.csv existing label — class balance: {existing_pct:.1f}% Well Received')

    # Row-for-row comparison, assuming both files preserve the same filtering order
    matches = (df_filtered['reception_label_75_reconstructed'].values == df_t4[TARGET_COL].values)
    match_rate = matches.mean() * 100
    print(f'Row-for-row label agreement: {match_rate:.2f}%')

    if match_rate > 99.5:
        print()
        print('VERIFIED — reconstructed filter and label align with the existing processed file.')
        print('Safe to proceed to alternative thresholds.')
        VERIFICATION_PASSED = True
    else:
        print()
        print('*** WARNING: row order likely does not match between this notebook\'s raw-data')
        print('*** reconstruction and games_features_t4.csv. This usually means NB02 applied an')
        print('*** additional sort/shuffle/dedup step not reproduced here. DO NOT proceed to')
        print('*** Section 4 until this is resolved — results would be silently misaligned.')
        VERIFICATION_PASSED = False


Reconstructed 75% threshold — class balance:
  Well Received     : 71.8%
  Not Well Received : 28.2%
  Expected (per NB01/paper): 71.8% / 28.2%

games_features_t4.csv existing label — class balance: 71.8% Well Received
Row-for-row label agreement: 100.00%

VERIFIED — reconstructed filter and label align with the existing processed file.
Safe to proceed to alternative thresholds.


## 4. Build Alternative-Threshold Labels

Only proceeds if Section 3's verification passed. We test five reception thresholds either side
of the current 75% cutoff. Structured features (`X`) are the existing T4 features from
`games_features_t4.csv`, unchanged, only the label varies.


In [4]:
assert VERIFICATION_PASSED, 'Verification failed in Section 3 — resolve before continuing.'

ALT_THRESHOLDS = [60, 65, 70, 75, 80, 85]

# Uses config.py's TIER_FEATURES[4] directly — verified via verify_tier_order.py.
feature_cols = TIER_FEATURES[4]
missing = [c for c in feature_cols if c not in df_t4.columns]
assert not missing, f'Columns missing from CSV: {missing}'
X = df_t4[feature_cols].values

labels_by_threshold = {}
for t in ALT_THRESHOLDS:
    label = (df_filtered[COL_LABEL].values >= t).astype(int)
    pct_positive = label.mean() * 100
    labels_by_threshold[t] = label
    marker = '  <- current design choice' if t == 75 else ''
    print(f'Threshold >= {t}%:  {pct_positive:.1f}% Well Received / {100-pct_positive:.1f}% Not Well Received{marker}')


Threshold >= 60%:  90.8% Well Received / 9.2% Not Well Received
Threshold >= 65%:  86.5% Well Received / 13.5% Not Well Received
Threshold >= 70%:  80.0% Well Received / 20.0% Not Well Received
Threshold >= 75%:  71.8% Well Received / 28.2% Not Well Received  <- current design choice
Threshold >= 80%:  60.0% Well Received / 40.0% Not Well Received
Threshold >= 85%:  46.1% Well Received / 53.9% Not Well Received


## 5. Retrain Model D at Each Threshold

Same architecture, same hyperparameters, same features. Only the label changes per threshold.
Each run uses its own stratified 80/20 split (`stratify=y` on that threshold's label, since the
class balance differs from one threshold to the next by design).


In [5]:
results = {}

for t in ALT_THRESHOLDS:
    y = labels_by_threshold[t]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
    )

    start = time.time()
    model = CatBoostClassifier(**CB_PARAMS)
    model.fit(X_train, y_train)
    elapsed = time.time() - start

    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    macro = f1_score(y_test, y_pred, average='macro')
    minor = f1_score(y_test, y_pred, pos_label=0)
    auc   = roc_auc_score(y_test, y_prob)
    acc   = accuracy_score(y_test, y_pred)

    results[t] = {
        'macro_f1': macro, 'minority_f1': minor, 'auc': auc, 'accuracy': acc,
        'pct_positive': y.mean() * 100, 'train_time': elapsed,
    }

    marker = '  <- baseline (matches paper)' if t == 75 else ''
    print(f'Threshold >= {t}%:  Macro F1={macro:.4f}  Minority F1={minor:.4f}  '
          f'AUC={auc:.4f}  Accuracy={acc:.4f}  ({elapsed:.1f}s){marker}')


Threshold >= 60%:  Macro F1=0.5705  Minority F1=0.1907  AUC=0.7470  Accuracy=0.9063  (15.7s)
Threshold >= 65%:  Macro F1=0.6185  Minority F1=0.3139  AUC=0.7597  Accuracy=0.8617  (17.3s)
Threshold >= 70%:  Macro F1=0.6319  Minority F1=0.3844  AUC=0.7363  Accuracy=0.7981  (20.1s)
Threshold >= 75%:  Macro F1=0.6690  Minority F1=0.5172  AUC=0.7394  Accuracy=0.7385  (17.5s)  <- baseline (matches paper)
Threshold >= 80%:  Macro F1=0.6613  Minority F1=0.6152  AUC=0.7299  Accuracy=0.6676  (16.4s)
Threshold >= 85%:  Macro F1=0.6438  Minority F1=0.7311  AUC=0.7445  Accuracy=0.6652  (16.9s)


## 6. Results Table and Class-Balance Comparison

Consolidated view, directly comparable to Table II's Model D row (Macro F1 = 0.6690 at the 75%
threshold).


In [6]:
print('=== RECEPTION-THRESHOLD SENSITIVITY — FULL RESULTS ===')
print(f'{"Threshold":>10} {"%% Positive":>11} {"Macro F1":>10} {"Minority F1":>12} '
      f'{"AUC":>8} {"Accuracy":>10} {"Delta vs 75%%":>13}')
print('-' * 78)

baseline_macro = results[75]['macro_f1']
for t in ALT_THRESHOLDS:
    r = results[t]
    delta = r['macro_f1'] - baseline_macro
    marker = ' <- current' if t == 75 else ''
    print(f'{t:>9}%% {r["pct_positive"]:>10.1f}%% {r["macro_f1"]:>10.4f} {r["minority_f1"]:>12.4f} '
          f'{r["auc"]:>8.4f} {r["accuracy"]:>10.4f} {delta:>+13.4f}{marker}')

print()
print(f'For reference, NB04 reports Model D at the 75%% threshold: Macro F1 = 0.6690')
print(f'This notebook\'s 75%% re-run: Macro F1 = {baseline_macro:.4f} '
      f'(should be close — small differences are expected from re-fitting a fresh model instance)')


=== RECEPTION-THRESHOLD SENSITIVITY — FULL RESULTS ===
 Threshold %% Positive   Macro F1  Minority F1      AUC   Accuracy Delta vs 75%%
------------------------------------------------------------------------------
       60%%       90.8%%     0.5705       0.1907   0.7470     0.9063       -0.0985
       65%%       86.5%%     0.6185       0.3139   0.7597     0.8617       -0.0505
       70%%       80.0%%     0.6319       0.3844   0.7363     0.7981       -0.0371
       75%%       71.8%%     0.6690       0.5172   0.7394     0.7385       +0.0000 <- current
       80%%       60.0%%     0.6613       0.6152   0.7299     0.6676       -0.0076
       85%%       46.1%%     0.6438       0.7311   0.7445     0.6652       -0.0252

For reference, NB04 reports Model D at the 75%% threshold: Macro F1 = 0.6690
This notebook's 75%% re-run: Macro F1 = 0.6690 (should be close — small differences are expected from re-fitting a fresh model instance)


## 7. Summary

This is the evidence needed to answer Reviewer 1's request for threshold justification. Two
patterns are worth writing into the paper regardless of which way the numbers land:

1. **How Macro F1 moves as the threshold shifts** — if performance is relatively stable across
   the tested range, that supports the current 75% choice being a reasonable, non-arbitrary
   point rather than one that happens to maximise the metric.
2. **How class balance shifts** — moving the threshold changes what fraction of games count as
   "Well Received." A threshold that produces a severely imbalanced split (e.g. >90% or <50%
   positive) is a weaker practical choice regardless of Macro F1, since it would leave very few
   examples of one class to learn from or evaluate on.

Suggested paper wording once you have the actual numbers: *"Reception thresholds from 60% to 85%
were tested by relabelling the dataset and retraining Model D with identical hyperparameters and
features. Macro F1 [rose/fell/remained stable] across this range ([X] to [Y]), with the class
balance shifting from [A]% to [B]% Well Received. The 75% threshold was retained because [reason
based on your actual results — e.g. it sits in the stable region / balances class sizes
reasonably / etc.]."*

Nothing in NB01–NB14 or in `games_features_t4.csv` was modified by running this notebook.
